# Đánh giá Retrieval: So sánh các phương pháp

## Mục tiêu

So sánh khả năng retrieval thực sự của:
1. **BM25** - baseline lexical retrieval (không dùng embedding)
2. **E5 pretrained** - E5-base chưa fine-tune
3. **E5 fine-tuned** - E5-base đã fine-tune trên tập train

Sử dụng **query tự nhiên** không chứa literal title của sản phẩm để đánh giá semantic retrieval thực chất.

## 1) Cài đặt thư viện

In [ ]:
# !pip -q install sentence-transformers rank-bm25 scikit-learn pandas numpy tqdm matplotlib seaborn

## 2) Setup đường dẫn

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/PhamMinhDan/llm_provider_benchmarking_ver2.git"
COLAB_REPO = Path("/content/llm_provider_benchmarking")

if COLAB_REPO.exists() and (COLAB_REPO / ".git").is_dir():
    subprocess.run(["git", "-C", str(COLAB_REPO), "pull", "--ff-only"], check=False)
elif not (COLAB_REPO / "embedding_project" / "data" / "ecommerce.csv").is_file():
    if COLAB_REPO.exists():
        shutil.rmtree(COLAB_REPO)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(COLAB_REPO)], check=True)

REPO_DIR = COLAB_REPO

from google.colab import drive
drive.mount("/content/drive")

CORPUS_CSV = REPO_DIR / "embedding_project/data/ecommerce.csv"
E5_PRETRAINED = "intfloat/e5-base"
E5_FINETUNED = Path("/content/drive/MyDrive/models/e5_base_finetuned_5000")
OUTPUT_DIR = REPO_DIR / "embedding_project/outputs/evaluation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Corpus: {CORPUS_CSV}")
print(f"E5 fine-tuned exists: {E5_FINETUNED.exists()}")

## 3) Tải dữ liệu corpus

In [ ]:
import pandas as pd
import ast

def parse_category(cat_str):
    """Parse category string thành list."""
    if pd.isna(cat_str):
        return []
    try:
        return ast.literal_eval(cat_str)
    except:
        return []

df = pd.read_csv(CORPUS_CSV)
df = df.dropna(subset=["product_id", "searchable_text"])
df = df.drop_duplicates(subset=["product_id"])
df["category_list"] = df["category"].apply(parse_category)
df["category_leaf"] = df["category_list"].apply(lambda x: x[-1] if x else "")

product_ids = df["product_id"].astype(str).tolist()
searchable_texts = df["searchable_text"].astype(str).tolist()
titles = df["title"].astype(str).tolist()
brands = df["brand"].astype(str).tolist()
categories = df["category_leaf"].astype(str).tolist()

print(f"Corpus: {len(product_ids)} sản phẩm")
print(f"Sample searchable_text:\n{searchable_texts[0][:200]}...")

## 4) Tạo tập query tự nhiên mới

**Quan trọng**: Không sử dụng query từ train/test/valid. Tạo query tự nhiên mới
dựa trên product attributes nhưng **không chứa literal title**.

In [ ]:
import random
import json

random.seed(42)

def extract_key_features(title, brand, category, description):
    """Trích xuất key features từ sản phẩm để tạo query tự nhiên."""
    features = []
    
    # Loại bỏ brand khỏi title để tránh trùng lặp
    clean_title = title.replace(brand, "").strip() if brand else title
    
    # Trích xuất từ mô tả (lấy phần đầu, ít thông tin nhất)
    desc_words = str(description).split()[:10] if pd.notna(description) else []
    
    return {
        "clean_title": clean_title,
        "brand": brand,
        "category": category,
        "desc_start": " ".join(desc_words)
    }

# Các template query tự nhiên (không chứa literal title)
QUERY_TEMPLATES = [
    # Mô tả công dụng
    "tìm kiếm {category} cho {use_case}",
    "cần mua {category} {feature}",
    "{category} tốt nhất dành cho {use_case}",
    
    # Hỏi về đặc điểm
    "{category} có đặc điểm {feature}",
    "cho tôi biết về {category} {feature}",
    "{category} loại nào tốt với {feature}",
    
    # Mua sắm
    "mua {category} {brand} ở đâu",
    "địa chỉ bán {category} {brand}",
    "shop bán {category} giá rẻ",
    
    # So sánh
    "so sánh các loại {category}",
    "{category} loại nào phổ biến nhất",
    "đánh giá {category} {brand}",
    
    # Xu hướng
    "{category} hot nhất hiện nay",
    "{category} được ưa chuộng nhất",
    "{category} trending 2024",
    
    # Khuyến nghị
    "gợi ý {category} cho người mới",
    "nên chọn {category} nào",
    "{category} nào được đánh giá cao",
    
    # Giá cả
    "{category} giá bao nhiêu",
    "{category} {brand} giá {price_range}",
    "tìm {category} {brand} giá tốt",
]

# Use cases
USE_CASES = [
    "người mới bắt đầu", "dân chuyên nghiệp", "sử dụng hàng ngày",
    "tập thể dục", "làm việc", "du lịch", "nấu ăn",
    "trang trí nhà", "sửa chữa", "học tập", "giải trí"
]

FEATURES = [
    "chất lượng cao", "giá rẻ", "bền", "đẹp", "nhẹ",
    "dễ sử dụng", "tiện lợi", "hiện đại", "cổ điển", "tối giản",
    "chống nước", "chống bụi", "an toàn", "thoải mái"
]

PRICE_RANGES = [
    "dưới 500k", "từ 500k đến 1 triệu", "trên 1 triệu",
    "giá rẻ", "giá trung bình", "cao cấp", "hàng xa xỉ"
]

def generate_natural_query(row):
    """Tạo query tự nhiên từ thông tin sản phẩm."""
    brand = str(row["brand"]) if pd.notna(row.get("brand")) else ""
    category = str(row["category_leaf"]) if pd.notna(row.get("category_leaf")) else "general"
    title = str(row.get("title", ""))
    
    # Chọn template ngẫu nhiên
    template = random.choice(QUERY_TEMPLATES)
    
    # Fill template
    use_case = random.choice(USE_CASES)
    feature = random.choice(FEATURES)
    price_range = random.choice(PRICE_RANGES)
    
    query = template.format(
        category=category,
        brand=brand,
        use_case=use_case,
        feature=feature,
        price_range=price_range
    )
    
    # Đảm bảo query có nghĩa
    query = query.strip()
    query = " ".join(query.split())  # normalize whitespace
    
    return query

# Chọn 500 sản phẩm ngẫu nhiên để tạo query (tránh overlap với train/test)
NUM_QUERIES = 500
sampled_indices = random.sample(range(len(df)), min(NUM_QUERIES, len(df)))
sampled_df = df.iloc[sampled_indices].reset_index(drop=True)

# Tạo query tự nhiên
natural_queries = []
for idx, row in sampled_df.iterrows():
    query = generate_natural_query(row)
    natural_queries.append({
        "query": query,
        "product_id": str(row["product_id"]),
        "title": str(row["title"]),
        "brand": str(row["brand"]) if pd.notna(row.get("brand")) else "",
        "category": str(row["category_leaf"])
    })

# Lưu query tự nhiên
NATURAL_QUERIES_FILE = OUTPUT_DIR / "natural_queries_500.jsonl"
with open(NATURAL_QUERIES_FILE, "w", encoding="utf-8") as f:
    for q in natural_queries:
        f.write(json.dumps(q, ensure_ascii=False) + "\n")

print(f"Đã tạo {len(natural_queries)} query tự nhiên")
print(f"Lưu tại: {NATURAL_QUERIES_FILE}")
print("\nSample queries:")
for i in range(5):
    print(f"  - {natural_queries[i]['query']}")
    print(f"    (target: {natural_queries[i]['title'][:50]}...)")

## 5) Triển khai BM25 Baseline

In [ ]:
from rank_bm25 import BM25Okapi
import numpy as np
import re

def preprocess_text(text):
    """Preprocess text cho BM25."""
    # Lowercase
    text = text.lower()
    # Remove special characters
    text = re.sub(r'[^\w\s]', ' ', text)
    # Tokenize
    tokens = text.split()
    # Remove short tokens
    tokens = [t for t in tokens if len(t) > 1]
    return tokens

def compute_bm25_retrieval(queries, corpus_texts, product_ids, k=100):
    """Compute BM25 retrieval for all queries."""
    print("Indexing corpus với BM25...")
    tokenized_corpus = [preprocess_text(doc) for doc in corpus_texts]
    bm25 = BM25Okapi(tokenized_corpus)
    
    print(f"Retrieving top-{k} cho {len(queries)} queries...")
    results = []
    for i, query in enumerate(queries):
        if i % 100 == 0:
            print(f"  Progress: {i}/{len(queries)}")
        tokenized_query = preprocess_text(query)
        scores = bm25.get_scores(tokenized_query)
        top_indices = np.argsort(scores)[::-1][:k]
        top_ids = [product_ids[idx] for idx in top_indices]
        results.append(top_ids)
    
    return results

# Chuẩn bị queries và labels
query_texts = [q["query"] for q in natural_queries]
labels = {q["query"]: {q["product_id"]} for q in natural_queries}

# BM25 retrieval
print("="*60)
print("BM25 RETRIEVAL")
print("="*60)
bm25_results = compute_bm25_retrieval(query_texts, searchable_texts, product_ids, k=100)

## 6) Triển khai E5 Pretrained Baseline

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

def normalize(vectors):
    return vectors / (np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12)

def compute_e5_retrieval(model_name, queries, corpus_texts, product_ids, k=100):
    """Compute E5 retrieval (pretrained hoặc fine-tuned)."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading model: {model_name} on {device}")
    model = SentenceTransformer(model_name)
    model.max_seq_length = 512
    
    print("Encoding corpus...")
    corpus_e5 = [f"passage: {t}" for t in corpus_texts]
    corpus_emb = normalize(
        model.encode(corpus_e5, batch_size=128, show_progress_bar=True, convert_to_numpy=True)
    )
    
    print("Encoding queries...")
    query_e5 = [f"query: {q}" for q in queries]
    query_emb = normalize(
        model.encode(query_e5, batch_size=128, show_progress_bar=True, convert_to_numpy=True)
    )
    
    print(f"Computing similarity và retrieval...")
    scores = query_emb @ corpus_emb.T
    
    results = []
    for i in range(len(queries)):
        top_indices = np.argsort(-scores[i])[:k]
        top_ids = [product_ids[idx] for idx in top_indices]
        results.append(top_ids)
    
    return results

# E5 Pretrained (chưa fine-tune)
print("="*60)
print("E5 PRETRAINED (chưa fine-tune)")
print("="*60)
e5_pretrained_results = compute_e5_retrieval(
    E5_PRETRAINED, query_texts, searchable_texts, product_ids, k=100
)

## 7) Triển khai E5 Fine-tuned

In [ ]:
if E5_FINETUNED.exists():
    print("="*60)
    print("E5 FINE-TUNED")
    print("="*60)
    e5_finetuned_results = compute_e5_retrieval(
        str(E5_FINETUNED), query_texts, searchable_texts, product_ids, k=100
    )
else:
    print("E5 fine-tuned model not found. Chỉ so sánh BM25 và E5 pretrained.")
    e5_finetuned_results = None

## 8) Tính toán Metrics

In [ ]:
def compute_metrics(retrieved_ids, labels, k_values=[1, 5, 10, 20, 50, 100]):
    """Compute Precision, Recall, MRR, NDCG."""
    metrics = {f"Recall@{k}": [] for k in k_values}
    metrics.update({f"Precision@{k}": [] for k in k_values})
    metrics["MRR"] = []
    metrics["NDCG@10"] = []
    
    for retrieved, query in zip(retrieved_ids, [q["query"] for q in natural_queries]):
        rel = labels.get(query, set())
        if not rel:
            continue
        
        # MRR
        rr = 0
        for i, rid in enumerate(retrieved[:max(k_values)]):
            if rid in rel:
                rr = 1.0 / (i + 1)
                break
        metrics["MRR"].append(rr)
        
        # NDCG@10
        dcg = 0
        for i in range(min(10, len(retrieved))):
            rel_score = 1 if retrieved[i] in rel else 0
            dcg += rel_score / np.log2(i + 2)
        idcg = sum(1 / np.log2(i + 2) for i in range(min(10, len(rel))))
        ndcg = dcg / idcg if idcg > 0 else 0
        metrics["NDCG@10"].append(ndcg)
        
        # Precision & Recall
        for k in k_values:
            top_k = set(retrieved[:k])
            tp = len(top_k & rel)
            precision = tp / k
            recall = tp / len(rel) if len(rel) > 0 else 0
            metrics[f"Precision@{k}"].append(precision)
            metrics[f"Recall@{k}"].append(recall)
    
    return {k: np.mean(v) for k, v in metrics.items()}

# Tính metrics cho tất cả methods
K_VALUES = [1, 5, 10, 20, 50, 100]

print("="*60)
print("METRICS COMPARISON")
print("="*60)

results = {}

# BM25
print("\nTính BM25 metrics...")
results["BM25"] = compute_metrics(bm25_results, labels, K_VALUES)

# E5 Pretrained
print("\nTính E5 Pretrained metrics...")
results["E5_Pretrained"] = compute_metrics(e5_pretrained_results, labels, K_VALUES)

# E5 Fine-tuned
if e5_finetuned_results is not None:
    print("\nTính E5 Fine-tuned metrics...")
    results["E5_Finetuned"] = compute_metrics(e5_finetuned_results, labels, K_VALUES)

# Display comparison
print("\n" + "="*80)
print("RESULTS SUMMARY (trên tập query tự nhiên - 500 queries)")
print("="*80)

import pandas as pd
comparison_df = pd.DataFrame(results).T
display(comparison_df.round(4))

## 9) So sánh chi tiết

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Recall curves
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

methods = list(results.keys())
colors = ["#e74c3c", "#3498db", "#2ecc71"]

# Plot 1: Recall@K
ax1 = axes[0]
for i, method in enumerate(methods):
    recalls = [results[method][f"Recall@{k}"] for k in K_VALUES]
    ax1.plot(K_VALUES, recalls, marker="o", label=method, color=colors[i % len(colors)])
ax1.set_xlabel("K")
ax1.set_ylabel("Recall")
ax1.set_title("Recall@K")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Precision@K
ax2 = axes[1]
for i, method in enumerate(methods):
    precisions = [results[method][f"Precision@{k}"] for k in K_VALUES]
    ax2.plot(K_VALUES, precisions, marker="s", label=method, color=colors[i % len(colors)])
ax2.set_xlabel("K")
ax2.set_ylabel("Precision")
ax2.set_title("Precision@K")
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: MRR & NDCG
ax3 = axes[2]
metrics_bar = ["MRR", "NDCG@10"]
x = np.arange(len(metrics_bar))
width = 0.25
for i, method in enumerate(methods):
    values = [results[method][m] for m in metrics_bar]
    ax3.bar(x + i*width, values, width, label=method, color=colors[i % len(colors)])
ax3.set_ylabel("Score")
ax3.set_title("MRR & NDCG@10")
ax3.set_xticks(x + width)
ax3.set_xticklabels(metrics_bar)
ax3.legend()
ax3.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "retrieval_comparison_natural_queries.png", dpi=150)
plt.show()

print(f"\nBiểu đồ lưu tại: {OUTPUT_DIR / 'retrieval_comparison_natural_queries.png'}")

## 10) Phân tích chi tiết: BM25 vs Semantic

In [ ]:
def analyze_retrieval_gap(bm25_results, semantic_results, labels, k=10):
    """Phân tích gap giữa BM25 và semantic retrieval."""
    bm25_hits = []
    semantic_hits = []
    both_hits = []
    only_bm25 = []
    only_semantic = []
    
    for i, (bm25_r, sem_r, query) in enumerate(zip(bm25_results, semantic_results, [q["query"] for q in natural_queries])):
        rel = labels.get(query, set())
        if not rel:
            continue
        
        bm25_topk = set(bm25_r[:k])
        sem_topk = set(sem_r[:k])
        
        bm25_hit = len(bm25_topk & rel) > 0
        sem_hit = len(sem_topk & rel) > 0
        
        bm25_hits.append(bm25_hit)
        semantic_hits.append(sem_hit)
        
        if bm25_hit and sem_hit:
            both_hits.append(i)
        elif bm25_hit and not sem_hit:
            only_bm25.append(i)
        elif not bm25_hit and sem_hit:
            only_semantic.append(i)
    
    return {
        "bm25_hit_rate": np.mean(bm25_hits),
        "semantic_hit_rate": np.mean(semantic_hits),
        "both_hit": len(both_hits),
        "only_bm25": len(only_bm25),
        "only_semantic": len(only_semantic),
        "indices": {
            "both": both_hits,
            "only_bm25": only_bm25,
            "only_semantic": only_semantic
        }
    }

print("="*60)
print("BM25 vs E5_PRETRAINED Analysis")
print("="*60)

gap_analysis = analyze_retrieval_gap(bm25_results, e5_pretrained_results, labels, k=10)
print(f"\nHit Rate @10:")
print(f"  BM25:           {gap_analysis['bm25_hit_rate']:.2%}")
print(f"  E5 Pretrained:  {gap_analysis['semantic_hit_rate']:.2%}")
print(f"\nOverlap Analysis:")
print(f"  Both hit:       {gap_analysis['both_hit']} queries ({gap_analysis['both_hit']/len(natural_queries):.1%})")
print(f"  Only BM25:      {gap_analysis['only_bm25']} queries ({gap_analysis['only_bm25']/len(natural_queries):.1%})")
print(f"  Only Semantic:  {gap_analysis['only_semantic']} queries ({gap_analysis['only_semantic']/len(natural_queries):.1%})")

if e5_finetuned_results is not None:
    print("\n" + "-"*60)
    print("BM25 vs E5_FINETUNED Analysis")
    print("-"*60)
    gap_ft = analyze_retrieval_gap(bm25_results, e5_finetuned_results, labels, k=10)
    print(f"\nHit Rate @10:")
    print(f"  BM25:           {gap_ft['bm25_hit_rate']:.2%}")
    print(f"  E5 Fine-tuned:  {gap_ft['semantic_hit_rate']:.2%}")
    print(f"\nOverlap Analysis:")
    print(f"  Both hit:       {gap_ft['both_hit']} queries ({gap_ft['both_hit']/len(natural_queries):.1%})")
    print(f"  Only BM25:      {gap_ft['only_bm25']} queries ({gap_ft['only_bm25']/len(natural_queries):.1%})")
    print(f"  Only Semantic:  {gap_ft['only_semantic']} queries ({gap_ft['only_semantic']/len(natural_queries):.1%})")

## 11) Lưu kết quả

In [ ]:
# Lưu kết quả chi tiết
import json

output_file = OUTPUT_DIR / "retrieval_baseline_comparison.json"
output_data = {
    "num_queries": len(natural_queries),
    "query_type": "natural_queries_no_literal_title",
    "metrics": {method: {k: float(v) for k, v in m.items()} for method, m in results.items()},
    "gap_analysis": {
        "bm25_vs_pretrained": gap_analysis,
    },
    "sample_queries": natural_queries[:10]  # Lưu 10 sample
}

if e5_finetuned_results is not None:
    output_data["gap_analysis"]["bm25_vs_finetuned"] = gap_ft

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print(f"Kết quả lưu tại: {output_file}")

# In bảng so sánh cuối cùng
print("\n" + "="*80)
print("FINAL COMPARISON - Natural Queries (không chứa literal title)")
print("="*80)
final_comparison = pd.DataFrame({
    "Method": methods,
    "Recall@10": [results[m]["Recall@10"] for m in methods],
    "Precision@10": [results[m]["Precision@10"] for m in methods],
    "MRR": [results[m]["MRR"] for m in methods],
    "NDCG@10": [results[m]["NDCG@10"] for m in methods],
    "Recall@50": [results[m]["Recall@50"] for m in methods],
    "Recall@100": [results[m]["Recall@100"] for m in methods],
})
display(final_comparison.round(4))

## 12) Kết luận

### Nhận xét quan trọng:

1. **BM25 baseline**: Nếu BM25 đạt Recall@10 cao (>70%), đây là dấu hiệu
   query có quá nhiều overlap từ vựng với corpus → không đánh giá được
   semantic retrieval thực sự.

2. **E5 pretrained vs fine-tuned**: Chênh lệch giữa hai model cho biết
   fine-tuning có hiệu quả hay không trên retrieval task thực sự.

3. **Gap BM25 vs Semantic**: 
   - Gap nhỏ → query-dependent vocabulary overlap
   - Gap lớn → semantic model thực sự capture được intent

### Khuyến nghị:

- Nếu BM25 Recall@10 > 80%: Query cần được thiết kế lại
- Nếu E5 fine-tuned không tốt hơn pretrained đáng kể: Cần cải thiện training data
- Nên chạy statistical significance test (Wilcoxon) để xác nhận improvement